In [ ]:
%idle_timeout 2880 # Tiempo máximo de inactividad
%glue_version 5.0 # Versión de AWS Glue
%worker_type G.1X # Tipo worker
%number_of_workers 5 # Número de workers para Spark

# Importamos librerías necesarias
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

BRONZE_BUCKET = "YOUR_BRONZE_BUCKET"
SILVER_BUCKET = "YOUR_SILVER_BUCKET"
  
sc = SparkContext.getOrCreate() # Obtenemos/Creamos el SparkContext
glueContext = GlueContext(sc) # Contexto de Glue
spark = glueContext.spark_session # Sesión de Spark
job = Job(glueContext) # Creamos el job de Glue

for year in range(2022, 2026):

    input_path = (
        f"s3://{BRONZE_BUCKET}/"
        f"LINKUSD/year={year}/linkusd{year}.csv"
    )

    output_path = (
        f"s3://{SILVER_BUCKET}/"
        f"LINKUSD/year={year}/"
    )

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(input_path)
    )

    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .save(output_path)
    )
job.commit() # Finalizamos el job de Glue
df = spark.read.parquet(
    f"s3://{SILVER_BUCKET}/LINKUSD/"
)